In [ ]:
import os
os.chdir("/content")
if not os.path.exists("llm-personality-conditioning"):
    !git clone -b new-direct-prompting https://github.com/beedle2017/llm-personality-conditioning.git
%cd /content/llm-personality-conditioning
!git pull
!pip install -q bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/direct_prompting

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import sys
sys.path[:0] = ["/content/llm-personality-conditioning/src",
                "/content/llm-personality-conditioning/scripts"]

from rewriting.llm import HFGenerator
import direct_prompting

generator = HFGenerator.load(load_4bit=True)
OUT = "/content/drive/MyDrive/direct_prompting"

In [ ]:
from direct_prompting import run_experiment

run_experiment(generator, f"{OUT}/pilot.jsonl", limit=10, levels=[1, 5])

In [ ]:
import pandas as pd
df = pd.read_json(f"{OUT}/pilot.jsonl", lines=True)
print(df.groupby(["condition", "level"])["score"].agg(["mean", "count"]))
print(df.refusal.mean(), df.extracted.isna().mean())
df[df.condition == "persona"][["item_id", "level", "text", "extracted", "score"]].head(10)

In [ ]:
run_experiment(generator, f"{OUT}/behaviorbench_dev.jsonl", split="development")

In [ ]:
run_experiment(generator, f"{OUT}/bfi.jsonl", source="bfi")

In [ ]:
run_experiment(generator, f"{OUT}/behaviorbench_test.jsonl", split="test", confirm_frozen=True)